# 🌳 Árboles de Decisión

**Módulo 03** | **Sesión 6** | **Duración estimada: 1h**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_04_arboles_decision.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Árboles de decisión | Comprender cómo funcionan los árboles de decisión para clasificación | Comprensión |
| 2 | Visualización | Visualizar e interpretar un árbol de decisión entrenado | Aplicación |
| 3 | Control de complejidad | Ajustar hiperparámetros para evitar sobreajuste | Aplicación |
| 4 | Random Forest | Comprender la idea de combinar múltiples árboles (ensemble) | Comprensión |
| 5 | Importancia de variables | Identificar y comunicar qué variables son más predictivas | Análisis |

## 💡 ¿Por qué es importante para tu práctica docente?

Los **árboles de decisión** son probablemente el modelo de machine learning más intuitivo que existe. Funcionan exactamente como un **diagrama de flujo**:

```
¿Asistencia > 80%?
├── Sí → ¿Materias inscritas <= 5?
│         ├── Sí → Sin riesgo
│         └── No → Riesgo moderado
└── No → ¿Tiene beca?
          ├── Sí → Riesgo moderado
          └── No → Alto riesgo
```

Esto los hace perfectos para:

- **Comunicar resultados** a personas no técnicas (decanos, directores, consejos)
- **Explicar decisiones** de forma transparente ("el modelo identificó a este estudiante como en riesgo porque...")
- **Enseñar machine learning** — los estudiantes los entienden inmediatamente

Además, cuando combinamos muchos árboles en un **Random Forest**, obtenemos modelos mucho más precisos.

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## 📖 Sección 1: ¿Qué es un árbol de decisión?

Un **árbol de decisión** es un modelo que toma decisiones mediante una serie de **preguntas de sí o no** sobre los datos.

### ¿Cómo funciona?

1. Empieza con todos los datos
2. Busca la **mejor pregunta** para dividir los datos (la que mejor separa las clases)
3. Divide los datos según la respuesta
4. Repite el proceso en cada subgrupo
5. Para cuando se cumple alguna condición (profundidad máxima, pocos datos, etc.)

### Anatomía de un árbol

| Término | Significado |
|---------|-------------|
| **Nodo raíz** | La primera pregunta (arriba del todo) |
| **Nodo interno** | Una pregunta intermedia |
| **Hoja** | La predicción final (no hace más preguntas) |
| **Profundidad** | Cuántos niveles de preguntas tiene |
| **Split (división)** | Cada pregunta que divide los datos en dos |

In [ ]:
# Cargar datos de rendimiento académico
df = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')

# Crear variable objetivo
df['en_riesgo'] = (df['promedio'] < 10).astype(int)
df['beca_num'] = df['beca'].astype(int)
df['trabaja_num'] = df['trabaja'].astype(int)

# Features
features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas', 'beca_num', 'trabaja_num']
X = df[features]
y = df['en_riesgo']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Datos de entrenamiento: {len(X_train)}')
print(f'Datos de prueba: {len(X_test)}')

---

## 🌲 Sección 2: Árbol para Clasificación

Entrenemos un árbol de decisión para clasificar estudiantes en riesgo y visualicemos las preguntas que hace.

In [ ]:
# Entrenar un árbol con profundidad limitada para que sea legible
arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)

# Evaluar
y_pred_arbol = arbol.predict(X_test)
acc = accuracy_score(y_test, y_pred_arbol)
print(f'Accuracy del árbol (profundidad=4): {acc:.4f} ({acc*100:.1f}%)')

In [ ]:
# Visualizar el árbol
plt.figure(figsize=(20, 10))
plot_tree(arbol, 
          feature_names=features,
          class_names=['Sin riesgo', 'En riesgo'],
          filled=True,
          rounded=True,
          fontsize=9,
          proportion=True)
plt.title('Árbol de Decisión — Riesgo Estudiantil (profundidad=4)', fontsize=14)
plt.tight_layout()
plt.show()

print('Lectura del árbol:')
print('  - Cada nodo muestra la pregunta que se hace')
print('  - "samples" indica qué porcentaje de datos llega a ese nodo')
print('  - "value" muestra la proporción de cada clase')
print('  - El color indica la clase dominante (más intenso = más seguro)')

---

## ✅ Sección 3: Ventajas y Desventajas

### Ventajas de los árboles

| Ventaja | Descripción |
|---------|-------------|
| **Interpretable** | Puedes explicar cada decisión como un diagrama de flujo |
| **No requiere escalado** | Funciona con datos en diferentes escalas sin normalizar |
| **Maneja datos mixtos** | Acepta variables numéricas y categóricas |
| **Captura no linealidades** | A diferencia de la regresión lineal, puede aprender relaciones complejas |

### Desventajas

| Desventaja | Descripción |
|------------|-------------|
| **Sobreajuste fácil** | Sin restricciones, el árbol memoriza los datos |
| **Inestable** | Pequeños cambios en los datos pueden generar árboles muy diferentes |
| **Menos preciso** | Generalmente inferior a modelos ensemble como Random Forest |

---

## 🎛️ Sección 4: Controlar la Complejidad

Los árboles de decisión tienden al **sobreajuste**: si los dejamos crecer sin límite, memorizan los datos de entrenamiento.

### Hiperparámetros para controlar la complejidad

| Parámetro | Qué controla | Efecto |
|-----------|-------------|--------|
| `max_depth` | Profundidad máxima del árbol | Menos profundidad → más simple |
| `min_samples_leaf` | Mínimo de muestras en cada hoja | Más muestras → más generalización |
| `min_samples_split` | Mínimo de muestras para dividir un nodo | Más muestras → menos divisiones |

Veamos la diferencia entre un árbol simple y uno complejo.

In [ ]:
# Comparar árbol simple vs complejo
arbol_simple = DecisionTreeClassifier(max_depth=2, random_state=42)
arbol_complejo = DecisionTreeClassifier(max_depth=10, random_state=42)

arbol_simple.fit(X_train, y_train)
arbol_complejo.fit(X_train, y_train)

# Evaluar en train y test
resultados = pd.DataFrame({
    'Modelo': ['Árbol simple (depth=2)', 'Árbol complejo (depth=10)'],
    'Accuracy Train': [
        accuracy_score(y_train, arbol_simple.predict(X_train)),
        accuracy_score(y_train, arbol_complejo.predict(X_train))
    ],
    'Accuracy Test': [
        accuracy_score(y_test, arbol_simple.predict(X_test)),
        accuracy_score(y_test, arbol_complejo.predict(X_test))
    ]
})

resultados['Diferencia'] = resultados['Accuracy Train'] - resultados['Accuracy Test']
resultados

In [ ]:
# Visualizar el sobreajuste: accuracy en train vs test para diferentes profundidades
profundidades = range(1, 16)
acc_train = []
acc_test = []

for d in profundidades:
    arbol_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    arbol_d.fit(X_train, y_train)
    acc_train.append(accuracy_score(y_train, arbol_d.predict(X_train)))
    acc_test.append(accuracy_score(y_test, arbol_d.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(profundidades, acc_train, 'o-', color='steelblue', label='Train', linewidth=2)
plt.plot(profundidades, acc_test, 's-', color='coral', label='Test', linewidth=2)
plt.xlabel('Profundidad del Árbol')
plt.ylabel('Accuracy')
plt.title('Sobreajuste: Accuracy en Train vs Test')
plt.legend(fontsize=12)
plt.xticks(profundidades)
plt.tight_layout()
plt.show()

print('Observa cómo:')
print('  - Train accuracy siempre sube (el árbol memoriza más)')
print('  - Test accuracy sube al principio pero luego se estanca o baja')
print('  - La profundidad óptima es donde test accuracy es máxima')

---

## 🌳🌳🌳 Sección 5: Random Forest

### La idea: muchos árboles votan juntos

Un **Random Forest** (Bosque Aleatorio) combina **cientos de árboles de decisión** para hacer una predicción:

1. Cada árbol se entrena con una **muestra aleatoria** de los datos
2. Cada árbol solo ve un **subconjunto aleatorio** de las variables
3. Para predecir, cada árbol "vota" y la clase con más votos gana

### ¿Por qué funciona mejor?

Imagina una decisión importante en la universidad. ¿Confiarías más en:
- La opinión de **un solo profesor**?
- O en el voto de **100 profesores**, cada uno con perspectivas diferentes?

El Random Forest aplica exactamente esta filosofía: la sabiduría del grupo supera la del individuo.

In [ ]:
# Entrenar Random Forest
bosque = RandomForestClassifier(n_estimators=100, random_state=42)
bosque.fit(X_train, y_train)

# Evaluar
y_pred_bosque = bosque.predict(X_test)
acc_bosque = accuracy_score(y_test, y_pred_bosque)

# Comparar con el árbol individual
comparacion = pd.DataFrame({
    'Modelo': ['Árbol (depth=4)', 'Random Forest (100 árboles)'],
    'Accuracy Test': [
        accuracy_score(y_test, y_pred_arbol),
        acc_bosque
    ]
})

print('Comparación de modelos:')
comparacion

In [ ]:
# Importancia de variables según Random Forest
importancias = pd.DataFrame({
    'Variable': features,
    'Importancia': bosque.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(importancias['Variable'], importancias['Importancia'], color='forestgreen')
plt.xlabel('Importancia')
plt.title('Importancia de Variables — Random Forest')
plt.tight_layout()
plt.show()

print('\nLas variables más importantes son las que el modelo usa más')
print('para tomar decisiones (dividir los nodos)')

---

## 🎓 Sección 6: Caso — Segmentación de Estudiantes

Vamos a usar el Random Forest para:
1. Identificar qué factores importan más en el riesgo académico
2. Predecir el riesgo de un estudiante hipotético nuevo
3. Generar una explicación legible de la predicción

In [ ]:
# Reporte completo del Random Forest
print('Reporte de Clasificación — Random Forest')
print('=' * 55)
print(classification_report(y_test, y_pred_bosque,
                            target_names=['Sin riesgo', 'En riesgo']))

In [ ]:
# Predecir para un estudiante hipotético nuevo
nuevo_estudiante = pd.DataFrame({
    'semestre': [3],
    'edad': [21],
    'asistencia_pct': [65.0],
    'materias_inscritas': [6],
    'beca_num': [0],
    'trabaja_num': [1]
})

prediccion = bosque.predict(nuevo_estudiante)[0]
probabilidad = bosque.predict_proba(nuevo_estudiante)[0]

print('Perfil del estudiante hipotético:')
print(f'  Semestre: 3 | Edad: 21 | Asistencia: 65%')
print(f'  Materias inscritas: 6 | Beca: No | Trabaja: Sí')
print()
print(f'Predicción: {"EN RIESGO" if prediccion == 1 else "Sin riesgo"}')
print(f'Probabilidad de riesgo: {probabilidad[1]*100:.1f}%')
print(f'Probabilidad sin riesgo: {probabilidad[0]*100:.1f}%')

In [ ]:
# Usar un árbol simple para generar explicación legible
# (Random Forest no es fácilmente interpretable, pero un árbol individual sí)
arbol_explicable = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol_explicable.fit(X_train, y_train)

# Obtener la ruta de decisión para el nuevo estudiante
ruta = arbol_explicable.decision_path(nuevo_estudiante)
nodo_ids = ruta.indices

# Reconstruir las decisiones
tree = arbol_explicable.tree_
print('Explicación del modelo (árbol simplificado):')
print('=' * 50)
for i, nodo in enumerate(nodo_ids[:-1]):
    feature_idx = tree.feature[nodo]
    umbral = tree.threshold[nodo]
    feature_name = features[feature_idx]
    valor = nuevo_estudiante[feature_name].values[0]
    if valor <= umbral:
        direccion = '≤'
    else:
        direccion = '>'
    print(f'  Paso {i+1}: ¿{feature_name} {direccion} {umbral:.1f}? → Sí (valor = {valor})')

pred_arbol = arbol_explicable.predict(nuevo_estudiante)[0]
print(f'\n  Conclusión: {"EN RIESGO" if pred_arbol == 1 else "Sin riesgo"}')

---

## ✏️ Ejercicios

### Ejercicio 1: Árbol para desempeño laboral

Usando `rrhh_nomina.csv`, entrena un árbol de decisión para predecir **alto desempeño** (evaluacion_desempeno >= 4).

1. Carga los datos y crea la variable objetivo binaria
2. Selecciona features: edad, antiguedad, salario_mensual_usd, ausencias_anuales
3. Entrena un DecisionTreeClassifier con max_depth=3
4. Visualiza el árbol con plot_tree
5. ¿Cuál es la primera pregunta que hace el árbol?

In [ ]:
# Ejercicio 1: Árbol para desempeño laboral
# Tu código aquí


### Ejercicio 2: DecisionTree vs RandomForest

Usando los datos de riesgo estudiantil:

1. Entrena un DecisionTreeClassifier con max_depth=5
2. Entrena un RandomForestClassifier con n_estimators=100
3. Compara accuracy, precision y recall de ambos en el conjunto de test
4. ¿Cuál modelo es mejor? ¿Por qué?

In [ ]:
# Ejercicio 2: Comparación DecisionTree vs RandomForest
# Tu código aquí


### Ejercicio 3: Variables importantes

Usando el Random Forest del Ejercicio 2 (o del modelo de la Sección 5):

1. Obtén las 3 variables más importantes con `feature_importances_`
2. ¿Coinciden con tu intuición como docente?
3. Si tuvieras que diseñar un programa de acompañamiento estudiantil basado en estos resultados, ¿en qué factores te enfocarías?

In [ ]:
# Ejercicio 3: Variables importantes
# Tu código aquí


---

## 📋 Resumen

| Concepto | Punto clave |
|----------|-------------|
| **Árbol de decisión** | Modelo que toma decisiones con preguntas sí/no, como un diagrama de flujo |
| **Interpretabilidad** | Se puede visualizar y explicar cada decisión |
| **Sobreajuste** | Sin límites, el árbol memoriza; controlar con max_depth, min_samples_leaf |
| **Random Forest** | Combina muchos árboles → más preciso y estable |
| **Importancia de variables** | Random Forest indica qué features son más predictivas |
| **Trade-off** | Árbol individual = interpretable; Random Forest = preciso |

## 📚 Referencias

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Capítulo 8: Tree-Based Methods.

2. Scikit-learn developers. (2024). *Decision Trees*. https://scikit-learn.org/stable/modules/tree.html

3. Scikit-learn developers. (2024). *Ensemble methods: Random Forests*. https://scikit-learn.org/stable/modules/ensemble.html#forest

4. Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5-32.

5. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Capítulo 6: Decision Trees & Capítulo 7: Ensemble Learning.